In [ ]:
import pandas as pd
import pickle
import logging
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

NUMERIC_FEATURES = [
    "area_sqm", "bedrooms", "living_rooms", "bathrooms",
    "floor", "total_floors", "has_elevator", "building_age_years",
    "subway_distance_m", "west_lake_distance_m", "green_ratio",
    "parking_ratio", "management_fee_yuan_per_sqm",
]

CATEGORICAL_FEATURES = [
    "district", "orientation", "decoration",
    "property_type", "developer", "subway_line", "school_district_tier",
]

LEAKAGE_COLS = [
    "listing_date", "listing_duration_days",
    "price_adjustments_count", "unit_price_yuan_per_sqm",
]

TARGET = "total_price_wan"
MODEL_PATH = "XGmodel.pkl"


def load_and_preprocess(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["subway_distance_m"] = df["subway_distance_m"].fillna(9999)
    df = df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns])
    return df


def build_pipeline() -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
            ("num", "passthrough", NUMERIC_FEATURES),
        ],
        remainder="drop",
    )

    xgb = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
    )

    return Pipeline(steps=[("preprocessor", preprocessor), ("model", xgb)])


def print_feature_importance(pipeline: Pipeline) -> None:
    ohe = pipeline.named_steps["preprocessor"].named_transformers_["cat"]
    cat_feature_names = ohe.get_feature_names_out(CATEGORICAL_FEATURES).tolist()
    all_feature_names = cat_feature_names + NUMERIC_FEATURES

    importances = pipeline.named_steps["model"].feature_importances_
    importance_df = (
        pd.DataFrame({"feature": all_feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    logger.info("\n--- Feature Importances ---")
    for _, row in importance_df.iterrows():
        bar = "█" * int(row["importance"] * 200)
        logger.info(f"  {row['feature']:<50} {row['importance']:.4f}  {bar}")


def train(dataset_path: str = "hangzhou_housing_dataset.csv") -> None:
    logger.info("Loading dataset: %s", dataset_path)
    df = load_and_preprocess(dataset_path)

    X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
    y = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    logger.info("Train size: %d | Test size: %d", len(X_train), len(X_test))

    pipeline = build_pipeline()
    logger.info("Training XGBoost pipeline...")
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    logger.info("MAE: %.2f 万 | R²: %.4f", mean_absolute_error(y_test, y_pred), r2_score(y_test, y_pred))

    with open(MODEL_PATH, "wb") as f:
        pickle.dump(pipeline, f)
    logger.info("Pipeline saved to %s", MODEL_PATH)

    print_feature_importance(pipeline)


if __name__ == "__main__":
    train()

2026-05-23 16:08:00,577 INFO Loading dataset: hangzhou_housing_dataset.csv
2026-05-23 16:08:00,619 INFO Train size: 2000 | Test size: 500
2026-05-23 16:08:00,620 INFO Training XGBoost pipeline...
2026-05-23 16:08:01,589 INFO MAE: 37.56 万 | R²: 0.9404
2026-05-23 16:08:01,611 INFO Pipeline saved to xgb_pipeline_v2.pkl
2026-05-23 16:08:01,622 INFO 
--- Feature Importances ---
2026-05-23 16:08:01,623 INFO   district_西湖区                                       0.1686  █████████████████████████████████
2026-05-23 16:08:01,625 INFO   district_上城区                                       0.1062  █████████████████████
2026-05-23 16:08:01,626 INFO   district_滨江区                                       0.0892  █████████████████
2026-05-23 16:08:01,627 INFO   area_sqm                                           0.0847  ████████████████
2026-05-23 16:08:01,629 INFO   district_拱墅区                                       0.0677  █████████████
2026-05-23 16:08:01,632 INFO   property_type_别墅 (Villa)              